In [1]:
# =========================================================
# κm(T,p) fitting for CO2 / O3 with ARTS — shape-safe + decreasing p_grid
# =========================================================
import numpy as np
import xarray as xr
from pathlib import Path
from scipy import constants as _C
import pyarts
from FluxSimulator import generate_gridded_field_from_profiles
pyarts.cat.download.retrieve()
import scipy
from scipy.sparse import diags
import cvxpy as cp
import pandas as pd
import os

# -----------------------------
# Constants
# -----------------------------
D   = 2.0
g   = 9.81
N_A        = _C.N_A
constant_k = _C.k                    # 1.380649e-23 J/K

M_DRY = 28.9647e-3
MOLAR_MASS = {"CO2": 44.0095e-3, "O3": 47.9982e-3}

SPECIES          = ["CO2", "O3"]
FORCING_SPECIES_ORDER = ["CO2", "CH4", "N2O", "O3", "CFC11", "CFC12"] # Placeholder for future implementation with forcing



In [2]:
# -----------------------------
# Small utils
# -----------------------------
def squeeze_1d(a):
    out = np.asarray(a).squeeze()
    if out.ndim != 1:
        raise ValueError(f"Expected 1D after squeeze; got {out.shape}")
    return out

def assert_strictly_decreasing(vec, name="vector"):
    v = squeeze_1d(vec).astype(float)
    if not np.all(np.diff(v) < 0):
        raise ValueError(f"{name} must be strictly decreasing; got first/last = {v[0]}, {v[-1]}")

def vmr_to_mmr(vmr_species, molar_mass_species, molar_mass_dry_air=M_DRY):
    vmr = squeeze_1d(vmr_species).astype(float)
    return vmr * (float(molar_mass_species) / float(molar_mass_dry_air))

def absorption_coeff_to_xsec(k_species, p_level, T_level, vmr_species):
    """
    k_s [m^-1] -> σ_s [m^2/molecule], N_s = p*VMR/(k_B*T)
    k_species: (n_wvn, n_level); p,T,vmr: (n_level,)
    """
    p   = squeeze_1d(p_level).astype(float)[None, :]
    T   = squeeze_1d(T_level).astype(float)[None, :]
    vmr = squeeze_1d(vmr_species).astype(float)[None, :]
    N_s = (p * vmr) / (constant_k * T)
    return np.asarray(k_species, float) / np.clip(N_s, 1e-300, None)

def xsec_to_mass_absorption(sigma_s, molar_mass_species):
    """σ_s [m^2/molecule] -> κ_m [m^2/kg]"""
    return (N_A / float(molar_mass_species)) * np.asarray(sigma_s, float)

# =====================
# CORE: k × Δz → τ (and cumulative)
# =====================
def tau_from_linear_k(k_level_wvn, z_levels_m, center='level'):
    """
    Level-centered linear extinction k [m^-1] + geometric Δz -> layer τ (dimensionless).
    k_level_wvn : (n_level, n_wvn)
    z_levels_m  : (n_level,)
    returns: tau_layer (n_layer, n_wvn), tau_cum_top (n_layer, n_wvn)
    """
    z = squeeze_1d(z_levels_m).astype(float)
    k = np.asarray(k_level_wvn, dtype=float)
    if k.shape[0] != z.size:
        raise ValueError(f"Level mismatch: k has {k.shape[0]} levels, z has {z.size}")
    dz = np.abs(np.diff(z))                                # (n_layer,)
    if center == 'level':
        k_layer = 0.5 * (k[:-1, :] + k[1:, :])            # (n_layer, n_wvn)
    elif center == 'layer':
        if k.shape[0] != dz.size:
            raise ValueError("For center='layer', k must be (n_layer, n_wvn)")
        k_layer = k
    else:
        raise ValueError("center must be 'level' or 'layer'")
    tau_layer = k_layer * dz[:, None]                     # dimensionless
    # top-down cumulative (TOA->surface); infer if z is descending or ascending
    top_to_surface = z[0] > z[-1]
    tau_for_csum   = tau_layer if top_to_surface else tau_layer[::-1, :]
    tau_cum_top    = np.cumsum(tau_for_csum, axis=0)
    tau_cum_top    = tau_cum_top if top_to_surface else tau_cum_top[::-1, :]
    return tau_layer, tau_cum_top
# -----------------------------
# ARTS runner (returns p_levels too)
# -----------------------------
def calc_lbl_rt(atmosphere, species=("CO2",), w_grid=None):
    """
    Returns:
      tau_arts         : (n_wvn, n_layer)
      absorption_coeff : (n_species, n_wvn, n_level)  [1/m]
      p_levels         : (n_level,) [Pa]  strictly decreasing (surface→TOA)
      z_levels_hyp     : (n_level,) [m]   hypsometric from p & T
      T_levels         : (n_level,) [K]
    """

    assert w_grid is not None and len(w_grid) > 0
    ws = pyarts.workspace.Workspace(verbosity=0)
    ws.water_p_eq_agendaSet()
    ws.gas_scattering_agendaSet()
    ws.PlanetSet(option="Earth")
    ws.verbositySetScreen(ws.verbosity, 0)
    ws.IndexSet(ws.stokes_dim, 1)
    ws.jacobianOff()
    ws.cloudboxOff()

    # spectroscopy
    sp_list = list(species)
    ws.abs_speciesSet(species=sp_list)
    ws.abs_lines_per_speciesReadSpeciesSplitCatalog(basename="lines/")
    ws.ReadXsecData(basename="xsec/")

    # wavenumber -> frequency
    ws.f_grid = pyarts.arts.convert.kaycm2freq(w_grid)
    ws.abs_lines_per_speciesCompact()
    ws.abs_lines_per_speciesCutoff(option="ByLine", value=750e9)
    ws.abs_lines_per_speciesNormalization(option="SFS")
    ws.abs_lines_per_speciesTurnOffLineMixing()

    ws.propmat_clearsky_agendaAuto(T_extrapolfac=1E99)
    ws.VectorSetConstant(ws.surface_scalar_reflectivity, 1, 0.0)

    # atmosphere
    ws.atm_fields_compact = atmosphere
    ws.AtmosphereSet1D()
    ws.AtmFieldsAndParticleBulkPropFieldFromCompact()
    ws.Extract(ws.z_surface, ws.z_field, 0)
    ws.surface_skin_t = ws.t_field.value[0,0,0]
    ws.vmr_field.value = ws.vmr_field.value.value.clip(min=0.0)

    ws.cloudboxSetFullAtm()
    ws.scat_data_checked = 1
    ws.Touch(ws.scat_data)
    ws.pnd_fieldZero()
    ws.sensorOff()
    ws.jacobianOff()

    ws.scat_data_checkedCalc()
    ws.atmfields_checkedCalc()
    ws.atmgeom_checkedCalc()
    ws.cloudbox_checkedCalc()
    ws.lbl_checkedCalc()

    # ----------------------------
    # linear absorption coefficient (robust axes handling)
    # ----------------------------
    ws.propmat_clearsky_fieldCalc()
    k_raw = np.asarray(ws.propmat_clearsky_field.value)  # 7-D
    
    shp  = k_raw.shape
    nf   = len(w_grid)
    nlev = int(np.asarray(ws.p_grid.value).size)
    ns   = len(sp_list)
    
    # Identify frequency and level axes unambiguously by their sizes
    cand_f = [i for i, s in enumerate(shp) if s == nf]
    cand_p = [i for i, s in enumerate(shp) if s == nlev]
    if len(cand_f) != 1 or len(cand_p) != 1:
        raise ValueError(f"Cannot identify freq/level axes from shape {shp} (nf={nf}, nlev={nlev}); "
                         f"found f={cand_f}, p={cand_p}")
    ax_f, ax_p = cand_f[0], cand_p[0]
    
    # Try the "true species axis" route if ns>1 and unique
    cand_s = [i for i, s in enumerate(shp) if s == ns]
    if ns > 1 and len(cand_s) == 1:
        ax_s = cand_s[0]
        # Reorder to (species, freq, level), then squeeze any remaining 1s
        k_re = np.moveaxis(k_raw, (ax_s, ax_f, ax_p), (0, 1, 2))
        k_re = np.asarray(k_re).squeeze()
        if k_re.ndim == 2:  # extremely degenerate path, ensure species axis exists
            k_re = k_re[None, :, :]
        if k_re.shape[0] != ns or k_re.shape[1] != nf or k_re.shape[2] != nlev:
            raise ValueError(f"Unexpected reordered shape {k_re.shape}; "
                             f"expected (ns={ns}, nf={nf}, nlev={nlev})")
        absorption_coeff = k_re  # (n_species, n_wvn, n_level)

    else:
        # Fallback for ns==1 (or ambiguous species axis among many 1s):
        # Move freq & level up front → (nf, nlev, ...), then squeeze the rest to 2-D
        k_fp = np.moveaxis(k_raw, (ax_f, ax_p), (0, 1))
        k_fp = np.squeeze(k_fp)              # expect (nf, nlev)
        if k_fp.ndim != 2 or k_fp.shape != (nf, nlev):
            raise ValueError(f"Fallback expected (nf, nlev) after squeeze; got {k_fp.shape} from {shp}")
        absorption_coeff = k_fp[None, :, :]  # add species axis → (1, nf, nlev)
    

    # ----------------------------
    # DISORT tau (layer optical thickness)
    # ----------------------------
    ws.StringSet(ws.iy_unit, "1")
    ws.disort_aux_vars = ["Layer optical thickness"]
    ws.spectral_irradiance_fieldDisort(nstreams=2, emission=1)
    tau_arts = np.asarray(ws.disort_aux.value[0].value)    # (n_wvn, n_layer)
    if tau_arts.shape[0] != nf:
        raise ValueError(f"tau freq dim {tau_arts.shape[0]} != len(w_grid) {nf}")

    # Grids
    p_levels = np.asarray(ws.p_grid.value, dtype=float).ravel()    # (n_level,)
    T_levels = np.asarray(ws.t_field.value, dtype=float)[:,0,0]    # (n_level,)

    # Hypsometric z with same nlev as p_levels/k
    R = 287.0; g = 9.81
    z_levels_hyp = np.zeros_like(p_levels, dtype=float)
    for i in range(1, p_levels.size):
        p1, p2 = p_levels[i-1], p_levels[i]
        Tbar = 0.5*(T_levels[i-1] + T_levels[i])
        z_levels_hyp[i] = z_levels_hyp[i-1] + (R*Tbar/g) * np.log(p1/p2)

    return tau_arts, absorption_coeff, p_levels, z_levels_hyp, T_levels

class _Temp1D:
    """Tiny adapter so flux_up/down can use len(temp) and temp.values[...]"""
    __slots__ = ("values",)
    def __init__(self, arr):
        a = np.asarray(arr, dtype=float)
        if a.ndim != 1:
            raise ValueError(f"temperature must be 1D; got {a.shape}")
        self.values = a
    def __len__(self):
        return self.values.shape[0]


def B_nu(T, nu):
    """
    Planck function
    
    In:
    T [K]: temperature
    nu [cm-1]: multiply all nu's by 100 to convert to m-1 in formula, then multiply
               B by 100 for units of W/m^2/sr/cm-1
        
    Out:
        Planck function in units of W/m^2/sr/cm-1
    """
    k_B = scipy.constants.k # Boltzmann constant
    h = scipy.constants.h # Planck constant
    c = scipy.constants.speed_of_light # speed of light in a vacuum
    
    return ((2*h*(c**2)*((100*nu)**3))/(np.exp((h*c*(100*nu))/(k_B*T)) - 1))*100


def flux_up(tau, temperature, fgrid):
    """
    For arrays ordered TOA to surface, take temperature on half-levels and optical depth on full levels and compute flux

    The function is fragile/janky: it assumes a single profile, and assumes a (vertical) ordering, and ignores
    xarray dimension/coordinate names. I'm sure it could be improved 

    The source function S at each level is Plank emission at the level temperature - this will be biased relative 
    to ARTS, which uses a linear-in-tau assumption (we could add this if need be) 
    """
    assert(tau.shape[1]+1 == len(temperature))
    D = 2 # Diffusivity factor - use what ARTS uses (Gaussian integration) 
    F = np.zeros(shape = (tau.shape[0], len(temperature)))
    F[:, -1] = B_nu(temperature.values[-1], fgrid)
    for i in np.arange(len(temperature)-2, -1, -1): 
        S = B_nu(temperature.values[i], fgrid) * (1 - np.exp(-D * tau[:, i]))
        F[:, i] = F[:, i+1] * np.exp(-D * tau[:, i]) + S

    return np.pi * F
    
def flux_down(tau, temperature, fgrid):
    
    assert tau.shape[1] + 1 == len(temperature)
    D = 2 # Diffusivity factor - use what ARTS uses (Gaussian integration) 
    F = np.zeros(shape = (tau.shape[0], len(temperature)))
    # TOA LW boundary condition: no incoming longwave from space
    F[:, 0] = 0.0
    # march downward through layers i = 0..n_layers-1 (between half-level i and i+1)
    for i in range(tau.shape[1]):
        # use lower-interface (i+1) temperature for a simple source approx
        S = B_nu(temperature.values[i+1], fgrid) * (1.0 - np.exp(-D * tau[:, i]))
        F[:, i+1] = F[:, i] * np.exp(-D * tau[:, i]) + S
    return np.pi * F


def derivative_matrix(nx):
    """
    Constructs the centered second-order accurate first-order derivative, equivalent to np.gradient
    """
    diagonals = [[-1./2.], [0], [1./2.]] # main diagonal elements
    offsets = [-1, 0, 1] # positions of the diagonal entries relative to the main diagonal

    # Call to the diags routine; note that diags returns a *representation* of the array;
    # to explicitly obtain its ndarray realisation, the call to .toarray() is needed.
    d1mat = diags(diagonals, offsets, shape=(nx,nx)).toarray()

    # We replace the first and last lines of d1mat with the proper
    # one-sided finite differences
    d1mat[0, :2] = np.array([-1, 1]) # first line
    d1mat[-1, -2:] = np.array([1, -1]) # last line

    # Return the final array divided by the grid spacing
    return d1mat

    
def cost_no_forcing(y_hat, y_ref, x_sup):
    """
    In: 
        y_hat: estimate; flattened array of (rows*scenarios*cols + gases*cols) shape, 
            where the last columns are forcings
            The order of the forcings should match that in the map function
            Here we're using CO2, CH4, N2O, O3, CFC11, CFC12
        y_ref: reference values/data 
        x_sup: supplementary data, here, heating rates and pressures.
    Out:
        cost_function: the value of the cost function
    """
    
    if isinstance(y_hat, xr.Dataset):
        # dummy return for checks 
        return cp.norm(y_ref.reference_forcing_co2.data - y_hat.reference_forcing_co2.data)
    
    # constants
    g = 9.81 # gravity (m/s^2)
    scaling = 3600*24 # heating rate conversion factor: seconds * minutes * hours in a day 

    ## choose columns with random variability in zenith angle, reflectivity.
    scenario_cols = np.random.randint(0, N_SCENARIOS, N_COLS, dtype = int) + N_SCENARIOS*np.arange(N_COLS)

    # constants for indexing through the flattened array
    end_fluxes_idx = N_LEVELS*N_COLS*N_SCENARIOS
    end_co2_idx = end_fluxes_idx + N_COLS*N_SCENARIOS
    end_ch4_idx = end_co2_idx + N_COLS*N_SCENARIOS
    end_n2o_idx = end_ch4_idx + N_COLS*N_SCENARIOS
    end_o3_idx = end_n2o_idx + N_COLS*N_SCENARIOS
    end_cfc11_idx = end_o3_idx + N_COLS*N_SCENARIOS

    # separate flux columns
    F_est = y_hat[:end_fluxes_idx]
    F_est = cp.reshape(F_est, [N_COLS*N_SCENARIOS, N_LEVELS], 'F')[scenario_cols, :]
    F_est = cp.reshape(F_est, [N_COLS*N_LEVELS], 'F')


    # separate forcing columns
    Force_est_co2 = y_hat[end_fluxes_idx:end_co2_idx]
    Force_est_co2 = Force_est_co2[scenario_cols]

    Force_est_ch4 = y_hat[end_co2_idx:end_ch4_idx]
    Force_est_ch4 = Force_est_ch4[scenario_cols]

    Force_est_n2o = y_hat[end_ch4_idx:end_n2o_idx]
    Force_est_n2o = Force_est_n2o[scenario_cols]

    Force_est_o3 = y_hat[end_n2o_idx:end_o3_idx]
    Force_est_o3 = Force_est_o3[scenario_cols]

    Force_est_cfc11 = y_hat[end_o3_idx:end_cfc11_idx]
    Force_est_cfc11 = Force_est_cfc11[scenario_cols]

    Force_est_cfc12 = y_hat[end_cfc11_idx:]
    Force_est_cfc12 = Force_est_cfc12[scenario_cols]

    # choose scenario subset in reference data
    y_ref = y_ref.isel(column = scenario_cols)
    x_sup = x_sup.isel(column = scenario_cols)

    # L2 norm of estimated flux error
    F_err = cp.norm((F_est - y_ref.reference_fluxes.data.reshape(-1)))

    # compute estimated heating rate from estimated fluxes
    H_est = -scaling*(cp.matmul(mat, cp.reshape(F_est, [N_COLS, N_LEVELS], 'F').T)*g)/cp.matmul(mat, x_sup.pressures.data.T)/1004

    # L2 norm of estimated heating rate error
    H_err = cp.norm((H_est.T-x_sup.reference_heating))

    # L2 norms of forcing estimates
    Force_err_co2 = cp.norm((Force_est_co2 - y_ref.reference_forcing_co2.data.reshape(-1)))

    Force_err_ch4 = cp.norm((Force_est_ch4 - y_ref.reference_forcing_ch4.data.reshape(-1)))

    Force_err_n2o = cp.norm((Force_est_n2o - y_ref.reference_forcing_n2o.data.reshape(-1)))

    Force_err_o3 = cp.norm((Force_est_o3 - y_ref.reference_forcing_o3.data.reshape(-1)))

    Force_err_cfc11 = cp.norm((Force_est_cfc11 - y_ref.reference_forcing_cfc11.data.reshape(-1)))

    Force_err_cfc12 = cp.norm((Force_est_cfc12 - y_ref.reference_forcing_cfc12.data.reshape(-1)))


    cost_function = F_ARRAY[0]*F_err + F_ARRAY[1]*H_err + F_ARRAY[2]*Force_err_co2 + F_ARRAY[3]*Force_err_ch4 + F_ARRAY[4]*Force_err_n2o + F_ARRAY[5]*Force_err_o3 + F_ARRAY[6]*Force_err_cfc11 + F_ARRAY[7]*Force_err_cfc12

    return cost_function


# -----------------------------
# Atmosphere builder (force decreasing p)
# -----------------------------
def make_isothermal_atmosphere_for_species(p_hl_Pa, T_const_K, species, vmr_value):
    """
    Build a 1D isothermal column at pressure half-levels p_hl_Pa with constant T,
    only the target species present (others zero). p_hl must be strictly decreasing.
    """
    p_hl = np.asarray(p_hl_Pa, float)
    # ensure strictly decreasing: surface->TOA
    if not np.all(np.diff(p_hl) < 0):
        p_hl = p_hl[::-1]
    assert_strictly_decreasing(p_hl, "p_hl_Pa")

    T_hl = np.full_like(p_hl, float(T_const_K))
    gases = {species: np.full_like(p_hl, float(vmr_value))}
    atmosphere = generate_gridded_field_from_profiles(
        p_hl, T_hl, gases=gases, particulates={}, z_field=None
    )
    return atmosphere

# -----------------------------
# κ sampling over designed T–p grid (shape-safe)
# -----------------------------
def generate_kappam_grid_for_species(species, wvn_cm1, p_hl_grid_Pa, T_set_K, vmr_value):
    """
    Returns:
      kappa_m : (n_wvn, n_p, n_T)  on the actual ARTS level grid
      P_used  : (n_p,) [Pa]        strictly decreasing
      T_used  : (n_T,) [K]
    """
    Ms = MOLAR_MASS[species]
    n_wvn = len(wvn_cm1)
    n_T   = len(T_set_K)
    kappa_list = []
    P_used = None

    for Tj in T_set_K:
        atm = make_isothermal_atmosphere_for_species(p_hl_grid_Pa, Tj, species, vmr_value)
        tau_arts, k_species_all, p_levels, z_hyp, T_levels = calc_lbl_rt(atm, species=(species,), w_grid=wvn_cm1)
        
        k_s = k_species_all[0, :, :]         # (n_wvn, n_level)
        Tlev = np.full_like(p_levels, float(Tj))    # isothermal for this run
        VMR  = np.full_like(p_levels, float(vmr_value))
        
        sigma = absorption_coeff_to_xsec(k_s, p_levels, Tlev, VMR)          # (n_wvn, n_level)
        kappa = xsec_to_mass_absorption(sigma, MOLAR_MASS[species])         # (n_wvn, n_level)
        kappa_list.append(kappa)

        if P_used is None:
            P_used = p_levels
        else:
            # enforce same vertical grid at all T
            if len(p_levels) != len(P_used) or np.max(np.abs(p_levels - P_used)) > 1e-6:
                raise RuntimeError("ARTS vertical grid changed across temperatures; "
                                   "ensure atmosphere builder yields consistent p-grid.")

    # stack over T: list of (n_wvn, n_level) -> (n_T, n_wvn, n_level) -> move axis
    kappa_arr = np.stack(kappa_list, axis=0)    # (n_T, n_wvn, n_level)
    kappa_arr = np.moveaxis(kappa_arr, 0, 2)    # (n_wvn, n_level, n_T)
    return kappa_arr, P_used, np.asarray(T_set_K, float)

# -----------------------------
# Log-polynomial fit κ(T,p) per wavenumber
# -----------------------------
def logpoly_features_mesh(p_Pa, T_K):
    """
    Build features on a full mesh (n_p, n_T) → (n_p*n_T, 6):
    [1, ln p, ln T, (ln p)^2, (ln T)^2, (ln p)(ln T)]
    """
    lp = np.log(np.asarray(p_Pa).reshape(-1, 1))
    lt = np.log(np.asarray(T_K).reshape(1, -1))
    LP = np.repeat(lp, lt.shape[1], axis=1)
    LT = np.repeat(lt, lp.shape[0], axis=0)
    X = np.stack([np.ones_like(LP), LP, LT, LP**2, LT**2, LP*LT], axis=-1)
    return X.reshape(-1, 6)

def fit_logpoly_per_wavenumber(kappa_m_wvn_pT, p_grid_Pa, T_set_K, ridge=1e-6):
    """
    kappa_m_wvn_pT: (n_wvn, n_p, n_T)  → coeffs: (n_wvn, 6)
    """
    n_wvn, n_p, n_T = kappa_m_wvn_pT.shape
    X  = logpoly_features_mesh(p_grid_Pa, T_set_K)
    Xt = X.T
    G  = Xt @ X
    G.flat[::7] += ridge
    Ginv_Xt = np.linalg.solve(G, Xt)
    coeffs = np.empty((n_wvn, 6), dtype=float)
    for i in range(n_wvn):
        y = np.log(np.clip(kappa_m_wvn_pT[i].reshape(-1), 1e-300, None))
        coeffs[i] = Ginv_Xt @ y
    return coeffs

def eval_logpoly_on_grid(coeffs_i, p_grid_Pa, T_set_K):
    """
    Evaluate a single-ν model on full (p×T) grid → (n_p, n_T)
    """
    lp = np.log(np.asarray(p_grid_Pa).reshape(-1, 1))
    lt = np.log(np.asarray(T_set_K).reshape(1, -1))
    X = np.stack([
        np.ones_like(lp @ np.ones_like(lt)),
        np.repeat(lp, lt.shape[1], axis=1),
        np.repeat(lt, lp.shape[0], axis=0),
        (np.repeat(lp, lt.shape[1], axis=1))**2,
        (np.repeat(lt, lp.shape[0], axis=0))**2,
        (np.repeat(lp, lt.shape[1], axis=1)) * (np.repeat(lt, lp.shape[0], axis=0)),
    ], axis=-1)   # (n_p, n_T, 6)
    return np.exp((X @ coeffs_i).astype(float))

def metrics_kappam_fit(kappa_true, kappa_pred):
    eps = 1e-30
    diff = kappa_pred - kappa_true
    abs_rms = float(np.sqrt(np.mean(diff**2)))
    rel = np.abs(diff) / np.maximum(np.abs(kappa_true), eps)
    return {"abs_rms": abs_rms,
            "rel_median": float(np.nanmedian(rel)),
            "rel_p95": float(np.nanpercentile(rel, 95))}


# =========================
# κ_m(T,p) EVALUATION on a column (log–poly(ln p, ln T) with cross term)
# =========================

def eval_kappam_levelwise_logpoly6(coeffs_wvn_6, p_levels_Pa, T_levels_K):
    """
    Evaluate κ_m(ν; T,p) at column levels for ALL ν at once.
    coeffs_wvn_6 : (n_wvn, 6)  [β0, β1 ln p, β2 ln T, β3 (ln p)^2, β4 (ln T)^2, β5 ln p ln T]
    p_levels_Pa  : (n_level,)  strictly decreasing (surface→TOA)
    T_levels_K   : (n_level,)
    Returns: κ_m  (n_wvn, n_level) in m^2/kg
    """
    p = np.asarray(p_levels_Pa, float)
    T = np.asarray(T_levels_K,  float)
    lp, lt = np.log(p), np.log(T)                       # (n_level,)
    X = np.stack([np.ones_like(lp), lp, lt, lp**2, lt**2, lp*lt], axis=1)  # (n_level, 6)
    # (n_level,6) @ (6,n_wvn) -> (n_level, n_wvn) -> transpose
    return np.exp(X @ coeffs_wvn_6.T).T


# =========================
# κ_m → linear k for a column (per-species, then sum)
# =========================

def build_linear_k_from_surrogate(kappa_coeffs_by_species,           # dict: name -> (n_wvn,6)
                                  species_order,                      # list[str], e.g., ["CO2","O3"]
                                  p_levels_Pa, T_levels_K,           # (n_level,), (n_level,)
                                  vmr_by_species,                    # dict: name -> (n_level,)
                                  molar_mass_by_species,             # dict: name -> kg/mol
                                  R_dry=287.0):
    """
    Returns:
      k_total_wvn_lev : (n_wvn, n_level) linear extinction [1/m]
      k_species       : dict name -> (n_wvn, n_level)
    """
    p = np.asarray(p_levels_Pa, float)
    T = np.asarray(T_levels_K,  float)
    rho_air = p / (R_dry * T)                            # (n_level,)  kg/m^3

    k_total = None
    k_per = {}

    for s in species_order:
        beta = kappa_coeffs_by_species[s]               # (n_wvn, 6)
        kappa = eval_kappam_levelwise_logpoly6(beta, p, T)  # (n_wvn, n_level)
        q_s = vmr_to_mmr(vmr_by_species[s], molar_mass_by_species[s])  # (n_level,)
        rho_s = rho_air * q_s                           # (n_level,)
        k_s = kappa * rho_s[None, :]                    # (n_wvn, n_level)
        k_per[s] = k_s
        k_total = k_s if k_total is None else (k_total + k_s)

    return k_total, k_per


# =========================
# Heights from (p,T)
# =========================

def hypsometric_heights_from_pT(p_hl_Pa, T_hl_K, z0=0.0, R_dry=287.0, g=9.81):
    """
    Hypsometric integration on half-levels (surface→TOA, strictly decreasing p).
    Returns z_hl [m] with z[0]=z0 at surface.
    """
    p = np.asarray(p_hl_Pa, float)
    T = np.asarray(T_hl_K,  float)
    if not np.all(np.diff(p) < 0):
        raise ValueError("p_hl_Pa must be strictly decreasing (surface→TOA).")
    z = np.empty_like(p, dtype=float)
    z[0] = z0
    for i in range(1, p.size):
        Tbar = 0.5 * (T[i-1] + T[i])
        z[i] = z[i-1] + (R_dry * Tbar / g) * np.log(p[i-1] / p[i])
    return z


# =========================
# Spectral → broadband with the two-stream
# =========================

def broadband_flux_from_two_stream_tau(tau_layer_wvn, T_hl_K, nu_cm1, weights, to_surface_order=True):
    """
    Use the existing two-stream routines (flux_up, flux_down) to compute broadband profiles.

    Inputs:
      tau_layer_wvn : (n_wvn, n_layer)   τ for each ν and layer (surface→TOA or TOA→surface; see below)
      T_hl_K        : (n_level,)         temperatures on half-levels
      nu_cm1        : (n_wvn,)
      weights       : (n_wvn,)           DDQ spectral weights
      to_surface_order : bool            If True, return profiles in surface→TOA order (match cost)
    Returns:
      F_up, F_down, F_net : (n_level,) broadband W/m^2, ordered per to_surface_order
    """
    # The two-stream expects arrays ordered TOA→surface.
    # If current data are surface→TOA, reverse to TOA→surface before calling; then reverse back if requested.
    # Assume caller provides surface→TOA (consistent with cost/derivative operator); adapt here:

    # reverse to TOA→surface for two-stream:
    tau_ts = tau_layer_wvn[:, ::-1]          # (n_wvn, n_layer)
    T_ts   = np.asarray(T_hl_K, float)[::-1]   # (n_level,)
    
    # sanity checks before calling the flux routines
    n_wvn = tau_ts.shape[0]
    n_lev = T_ts.size
    assert tau_ts.shape[1] + 1 == n_lev, (
        f"tau has {tau_ts.shape[1]} layers, but T has {n_lev} levels; "
        "need n_levels = n_layers + 1"
    )
    assert n_wvn == len(nu_cm1) == len(weights), (
        f"ν/weight mismatch: tau has {n_wvn} spectra, "
        f"|nu|={len(nu_cm1)}, |W|={len(weights)}"
    )
    
    # wrap T so flux_up/down can call len(...) and index .values[...]
    T_obj = _Temp1D(T_ts)
    
    Fu_spec = flux_up(tau_ts,   T_obj, nu_cm1)  # (n_wvn, n_level)
    Fd_spec = flux_down(tau_ts, T_obj, nu_cm1)  # (n_wvn, n_level)

    # Integrate with DDQ weights (assumed 1D over ν)
    w = np.asarray(weights, float).reshape(-1, 1)   # (n_wvn,1)
    Fu = np.sum(w * Fu_spec, axis=0)                # (n_level,)
    Fd = np.sum(w * Fd_spec, axis=0)                # (n_level,)
    Fnet = Fu - Fd

    if to_surface_order:
        Fu, Fd, Fnet = Fu[::-1], Fd[::-1], Fnet[::-1]
    return Fu, Fd, Fnet

def kappa_m_to_linear(kappa_m_s, p_level, T_level, mmr_species, R=287.0):
    """
    κ_m,s [m^2/kg] + species mass density ρ_s -> linear k_s [m^-1]
    ρ_air = p / (R * T);  ρ_s = q_s * ρ_air;  k_s = κ_m,s * ρ_s
    Broadcast over (n_wvn, n_level)
    """
    p = squeeze_1d(p_level).astype(float)
    T = squeeze_1d(T_level).astype(float)
    q = squeeze_1d(mmr_species).astype(float)
    rho_air = (p / (R * T))[None, :]               # kg/m^3 (level)
    rho_s   = q[None, :] * rho_air                 # kg/m^3
    return np.asarray(kappa_m_s, dtype=float) * rho_s
    

def ensure_surface_to_toa_order(profile, species_order):
    """
    Return p_hl, T_hl, vmr (dict) all sorted to surface→TOA (strictly decreasing p).
    Does NOT mutate `profile`.
    """
    p_hl = np.asarray(profile.pressure_hl.values, float)
    T_hl = np.asarray(profile.temperature_hl.values, float)

    # Build VMR dict first (so we can re-index consistently)
    vmr = {}
    for s in species_order:
        key = f"{s.lower()}_mole_fraction_hl"
        vmr[s] = np.asarray(profile[key].values, float)

    # If not already strictly decreasing, reindex everything by the same ord_idx
    if not np.all(np.diff(p_hl) < 0):
        ord_idx = np.argsort(p_hl)[::-1]
        p_hl = p_hl[ord_idx]
        T_hl = T_hl[ord_idx]
        for s in species_order:
            vmr[s] = vmr[s][ord_idx]

    return p_hl, T_hl, vmr

def make_y_hat_with_species_swaps(
    target_species_list,          # e.g., ["CO2","O3"]  (case-insensitive)
    gas_names,                    # list of ALL gases in the LBL calc (UPPERCASE)
    k_species_lbl,                # (n_spec, n_wvn, n_level) linear k from ARTS LBL
    p_hl, T_hl, z_levels,         # 1-D half-level p [Pa], T [K], geometric z [m]
    nu_cm1, weights,              # DDQ ν and weights
    kappa_coeffs_dict,            # {"CO2": coeffs_per_ν, "O3": coeffs_per_ν, ...}
    molar_mass_dict,              # {"CO2": kg/mol, "O3": kg/mol, ...}
    vmr_dict,                     # {"CO2": VMR(level,), "O3": VMR(level,), ...} (surface→TOA)
    col
):
    """
    Build y_hat for the cost: full mixture where listed species are replaced
    by surrogate κ_m(T,p); all other species remain at their LBL linear-k.
    Returns:
      y_hat: flattened vector [F_net(levels), zeros_for_forcings]
    """
    # Start from total LBL linear k
    k_total_wvn_lev = np.sum(k_species_lbl, axis=0).copy()   # (n_wvn, n_level)

    # Normalize species keys
    target_upper = [s.upper() for s in target_species_list]

    # Map gas name -> index into k_species_lbl
    name_to_idx = {g: i for i, g in enumerate(gas_names)}

    for s in target_upper:
        if s not in name_to_idx:
            raise ValueError(f"Target species {s} not in LBL gas list {gas_names}")
        if s not in kappa_coeffs_dict:
            raise ValueError(f"No κ_m coefficients provided for species {s}")
        if s not in molar_mass_dict:
            raise ValueError(f"No molar mass for species {s}")
        if s not in vmr_dict:
            raise ValueError(f"No VMR profile available for species {s}")

        # Remove LBL contribution for this species
        s_idx = name_to_idx[s]
        k_total_wvn_lev -= k_species_lbl[s_idx, :, :]                  # (n_wvn, n_level)

        # Evaluate κ_m on column grid and convert to linear k
        #kappa_m_s = eval_kappam_levelwise_logpoly6(                    # (n_wvn, n_level)
        #    kappa_coeffs_dict[s], p_hl, T_hl
        #)
        q_s = vmr_to_mmr(vmr_dict[s], molar_mass_dict[s])              # (n_level,)
        k_rec_s = kappa_m_to_linear(kappa_coeffs_dict[s][:,col,:], p_hl, T_hl, q_s)        # (n_wvn, n_level)

        # Add surrogate species back into the total
        k_total_wvn_lev += k_rec_s

    # τ from k and then broadband flux with DDQ weights
    tau_layer_wvn, _ = tau_from_linear_k(k_total_wvn_lev.T, z_levels, center='level')
    tau_layer_wvn = tau_layer_wvn.T

    Fu, Fd, Fnet = broadband_flux_from_two_stream_tau(
        tau_layer_wvn, T_hl, nu_cm1, weights, to_surface_order=True
    )

    # Pack cost vector: net fluxes (levels) + 6 zeros for forcing slots
    y_hat = np.concatenate([Fnet.reshape(-1), np.zeros(6)], axis=0)
    return y_hat


def _discover_all_gases(profile):
    """
    Return sorted list of GAS NAMES (UPPERCASE) found in the CKDMIP column.
    We detect gases by *_mole_fraction_hl or *_mole_fraction_fl.
    """
    species = set()
    for v in profile.variables:
        if v.endswith("_mole_fraction_hl") or v.endswith("_mole_fraction_fl"):
            base = v.split("_mole_fraction")[0]
            species.add(base.upper())
    return sorted(species)


def build_reference_targets_from_lbl_fullmix(
    profile,
    nu_cm1,
    weights,
    N_LEVELS,
    mat,
    N_COLS=1,
    N_SCENARIOS=1,
):
    """
    For ONE CKDMIP column:
      - build full-mixture ARTS atmosphere with all available gases
      - run LBL on DDQ points (nu_cm1)
      - compute total k, tau, DDQ-weighted broadband fluxes and heating
      - return:
          y_ref, x_sup       (for cost_no_forcing)
          gas_names,
          k_species_lbl,     (n_spec, n_wvn, n_level)
          p_hl, T_hl,        (N_LEVELS,)
          z_levels,          (N_LEVELS,)
          vmr_dict           ({GAS: vmr(level,)})
    This is the per-column baseline used both for:
      - reference (LBL) cost
      - species-swap surrogate evaluation
    """

    # 1) Identify which gases exist in this column
    gas_names = _discover_all_gases(profile)
    if not gas_names:
        raise ValueError("No gas mole fraction fields found in profile.")

    # 2) Get aligned, surface→TOA p, T, and VMR for all gases
    #    ensure_surface_to_toa_order must:
    #      - read pressure_hl, temperature_hl from this column
    #      - for each gas in gas_names, read *_mole_fraction_hl (or from *_fl)
    #      - apply the SAME sort index so p,T,VMRs stay aligned
    #      - return p_hl [Pa], T_hl [K], and vmr_dict[GAS] (1D, length N_LEVELS)
    p_hl, T_hl, vmr_dict = ensure_surface_to_toa_order(profile, gas_names)

    # Sanity: N_LEVELS must match this column
    if p_hl.size != N_LEVELS:
        raise ValueError(
            f"N_LEVELS={N_LEVELS} does not match column half_levels={p_hl.size}"
        )

    # 3) Build ARTS atmosphere with the FULL mixture for this column
    atmosphere = generate_gridded_field_from_profiles(
        p_hl,
        T_hl,
        gases={g: vmr_dict[g] for g in gas_names},
        particulates={},
        z_field=None,
    )

    # 4) Run LBL once for this full mixture
    #    calc_lbl_rt must:
    #      - use 'species=tuple(gas_names)'
    #      - return:
    #           tau_arts: (n_wvn, n_layer)
    #           k_species_lbl: (n_spec, n_wvn, n_level)  linear k [1/m]
    #           p_levels: (n_level,)
    #           z_levels: (n_level,)
    #           T_levels: (n_level,)
    tau_arts, k_species_lbl, p_levels, z_levels, T_levels = calc_lbl_rt(
        atmosphere,
        species=tuple(gas_names),
        w_grid=nu_cm1,
    )

    # Basic consistency checks
    n_wvn = nu_cm1.size
    if k_species_lbl.shape[0] != len(gas_names):
        raise ValueError(
            f"Expected {len(gas_names)} species in k_species_lbl, got {k_species_lbl.shape[0]}"
        )
    if k_species_lbl.shape[1] != n_wvn:
        raise ValueError(
            f"Expected n_wvn={n_wvn} in k_species_lbl, got {k_species_lbl.shape[1]}"
        )
    if k_species_lbl.shape[2] != N_LEVELS:
        raise ValueError(
            f"Expected N_LEVELS={N_LEVELS} in k_species_lbl, got {k_species_lbl.shape[2]}"
        )

    # 5) Sum over species to total k(ν,level)
    k_total_wvn_lev = np.sum(k_species_lbl, axis=0)  # (n_wvn, n_level)

    # 6) Convert k -> τ per layer, then DDQ broadband flux
    tau_layer_wvn, _ = tau_from_linear_k(
        k_total_wvn_lev.T,  # (n_level, n_wvn)
        z_levels,
        center="level",
    )
    tau_layer_wvn = tau_layer_wvn.T  # (n_wvn, n_layer)

    Fu_ref, Fd_ref, Fnet_ref = broadband_flux_from_two_stream_tau(
        tau_layer_wvn,
        T_hl,
        nu_cm1,
        weights,
        to_surface_order=True,
    )  # each (N_LEVELS,)

    # 7) Heating rate using EXACTLY the same operator/constants as cost_no_forcing
    F_ref_2D = np.tile(Fnet_ref.reshape(N_LEVELS, 1), (1, N_COLS))  # (N_LEVELS, N_COLS)
    p_2D     = np.tile(p_hl.reshape(N_LEVELS, 1),   (1, N_COLS))    # (N_LEVELS, N_COLS)

    H_ref_2D = -86400.0 * (mat @ F_ref_2D) * 9.8 / (mat @ p_2D) / 1004.0  # (N_LEVELS, N_COLS)

    # 8) Pack y_ref / x_sup in shapes expected by cost_no_forcing
    level  = np.arange(N_LEVELS)
    column = np.arange(N_COLS)

    y_ref = xr.Dataset(
        data_vars=dict(
            reference_fluxes        =(("level", "column"), F_ref_2D),
            reference_forcing_co2   =(("column",), np.zeros(N_COLS)),
            reference_forcing_ch4   =(("column",), np.zeros(N_COLS)),
            reference_forcing_n2o   =(("column",), np.zeros(N_COLS)),
            reference_forcing_o3    =(("column",), np.zeros(N_COLS)),
            reference_forcing_cfc11 =(("column",), np.zeros(N_COLS)),
            reference_forcing_cfc12 =(("column",), np.zeros(N_COLS)),
        ),
        coords=dict(level=level, column=column),
    )

    x_sup = xr.Dataset(
        data_vars=dict(
            pressures        =(("column", "level"), p_2D.T),     # (N_COLS, N_LEVELS)
            reference_heating=(("column", "level"), H_ref_2D.T), # (N_COLS, N_LEVELS)
        ),
        coords=dict(level=level, column=column),
    )

    # 9) Return reference datasets + all tensors needed by species-swap
    return y_ref, x_sup, (gas_names, k_species_lbl, p_hl, T_hl, z_levels, vmr_dict)




def ckdmip_envelope(ds):
    # Use ALL columns/levels to define the envelope
    P = ds["pressure_hl"].values  # (column, half_level)
    T = ds["temperature_hl"].values
    p_all = np.asarray(P, float).ravel()
    t_all = np.asarray(T, float).ravel()
    p_all = p_all[np.isfinite(p_all) & (p_all > 0.0)]
    t_all = t_all[np.isfinite(t_all)]
    return float(p_all.min()), float(p_all.max()), float(t_all.min()), float(t_all.max())

def make_sampling_grids_from_envelope(
    ds,
    nT=9,
    nP=55,
    T_pad=0.08,      # 8% typical; MUST be a fraction (0–1)
    p_pad=0.08,      # 8% typical; MUST be a fraction (0–1)
    p_top_floor=1e-2 # Pa, allow going almost to CKDMIP min
):
    """
    Build:
      - T_set: ascending, padded by T_pad fraction
      - p_hl_grid: strictly decreasing (surface→TOA), padded by p_pad fraction
    """
    p_min, p_max, t_min, t_max = ckdmip_envelope(ds)

    # --- Temperature grid (ascending), padded fractionally, but keep > 0K
    T_lo = max(t_min * (1.0 - T_pad), 100.0)  # keep well above 0K for safety
    T_hi = t_max * (1.0 + T_pad)
    T_set = np.linspace(T_lo, T_hi, int(nT))

    # --- Pressure bounds with fractional padding
    p_surf = p_max * (1.0 + p_pad)           # near surface (high pressure)
    p_top  = max(p_min * (1.0 - p_pad), p_top_floor)  # TOA lower bound, honor floor

    # Geometric spacing from low→high then reverse -> strictly decreasing
    p_hl_grid = np.geomspace(p_top, p_surf, int(nP))[::-1]

    # Enforce strict decrease (guard against any float ties)
    for i in range(1, p_hl_grid.size):
        if not (p_hl_grid[i] < p_hl_grid[i-1]):
            p_hl_grid[i] = np.nextafter(p_hl_grid[i-1], 0.0)

    # Final sanity
    if np.any(T_set <= 0.0):
        raise ValueError("T_set contains non-positive temperatures. Use fractional pads (e.g., 0.05).")
    if not np.all(np.diff(p_hl_grid) < 0.0):
        raise ValueError("p_hl_grid must be strictly decreasing (surface→TOA).")

    return T_set, p_hl_grid

def report_sampling_vs_envelope(ds, T_set, p_hl_grid, p_top_floor=1e-2, eps_frac=1e-3):
    p_min, p_max, t_min, t_max = ckdmip_envelope(ds)
    print(f"CKDMIP envelope: p∈[{p_min:.2e},{p_max:.2e}] Pa, T∈[{t_min:.1f},{t_max:.1f}] K")
    print(f"Sampling grids : p∈[{p_hl_grid.min():.2e},{p_hl_grid.max():.2e}] Pa (surf→TOA), "
          f"T∈[{T_set.min():.1f},{T_set.max():.1f}] K")

    # Coverage (inside-or-equal)
    cover_p_hi = (p_hl_grid.max() >= p_max * (1 - eps_frac))
    cover_p_lo = (p_hl_grid.min() <= max(p_min, p_top_floor) * (1 + eps_frac))
    cover_T_lo = (T_set.min() <= t_min * (1 + eps_frac))
    cover_T_hi = (T_set.max() >= t_max * (1 - eps_frac))

    # Outside flags (strictly beyond by a tiny margin)
    outside_p_hi = (p_hl_grid.max() > p_max * (1 + eps_frac))
    outside_p_lo = (p_hl_grid.min() < p_min * (1 - eps_frac)) or (p_hl_grid.min() < p_top_floor * (1 - eps_frac))
    outside_T_lo = (T_set.min() < t_min * (1 - eps_frac))
    outside_T_hi = (T_set.max() > t_max * (1 + eps_frac))

    if cover_p_hi and cover_p_lo and cover_T_lo and cover_T_hi:
        if any([outside_p_hi, outside_p_lo, outside_T_lo, outside_T_hi]):
            print("✅ Sampling covers CKDMIP and extends a little outside.")
        else:
            print("✅ Sampling exactly covers the CKDMIP envelope (no outside padding).")
    else:
        print("⚠️ Sampling does not fully cover the CKDMIP envelope. Increase padding or nT/nP.")


def coerce_coeffs_wvn_by_features(coeffs_raw, nu_used, nu_target):
    """
    Ensure coeffs are (n_wvn_target, n_features) and aligned to nu_target.
    - coeffs_raw: 2D array, either (n_wvn_used, n_features) or (n_features, n_wvn_used)
    - nu_used:    1D array of wavenumbers the coeffs were trained on
    - nu_target:  1D array of wavenumbers to evaluate on (e.g., ddq.S)
    """
    C = np.asarray(coeffs_raw)
    if C.ndim != 2:
        raise ValueError(f"coeffs must be 2D, got {C.shape}")

    # Orient rows = wavenumbers
    if C.shape[0] == len(nu_used):
        C_wf = C                      # (n_wvn_used, n_feat)
    elif C.shape[1] == len(nu_used):
        C_wf = C.T                    # (n_wvn_used, n_feat)
    else:
        raise ValueError(
            f"coeffs shape {C.shape} inconsistent with len(nu_used)={len(nu_used)}"
        )

    # Reindex rows to match nu_target order (exact match to within tolerance)
    if np.array_equal(nu_used, nu_target):
        return C_wf

    idx = np.array([np.argmin(np.abs(nu_used - v)) for v in nu_target], dtype=int)
    if not np.allclose(nu_used[idx], nu_target, rtol=0, atol=1e-6):
        raise ValueError("Coefficient ν grid does not match target DDQ ν grid within tolerance.")
    return C_wf[idx, :]


# ---- Pretty-print fitted log–poly equations per wavenumber ----

LOGPOLY6_TERMS = ["1", "ln p", "ln T", "(ln p)^2", "(ln T)^2", "(ln p)(ln T)"]

def _coefs_orient_to_wvn_rows(coefs, nu_cm1):
    """
    Ensure coeffs are shaped (n_wvn, 6). Accepts (6, n_wvn) or (n_wvn, 6).
    """
    C = np.asarray(coefs)
    nnu = len(nu_cm1)
    if C.ndim != 2:
        raise ValueError(f"coeffs must be 2-D; got {C.shape}")
    if C.shape == (nnu, 6):
        return C
    if C.shape == (6, nnu):
        return C.T
    raise ValueError(f"Unexpected coeffs shape {C.shape}; expected (n_wvn,6) or (6,n_wvn) with n_wvn={nnu}")

def format_logpoly6_equation_row(a, nu, species, cfmt="{:+.6e}"):
    """
    a: length-6 array of coefficients [a0..a5]
    Returns a single Markdown-formatted string for this ν.
    """
    a0, a1, a2, a3, a4, a5 = [cfmt.format(x) for x in a]
    eq = (f"ln κ_m(ν={nu:.2f} cm⁻¹; {species}) = "
          f"{a0} + {a1}·ln p + {a2}·ln T + {a3}·(ln p)² + {a4}·(ln T)² + {a5}·(ln p)(ln T)  "
          f"<!-- units: κ_m[m²/kg], p[Pa], T[K] -->")
    return eq

def print_logpoly6_equations(coeffs, nu_cm1, species, sample=None, to_file=None, cfmt="{:+.6e}"):
    """
    coeffs : (n_wvn,6) or (6,n_wvn) array from the fit
    nu_cm1 : (n_wvn,) wavenumber grid (cm^-1)
    species: 'CO2' or 'O3'
    sample : None -> print all; or int -> print evenly spaced subset of that many
             or iterable of indices
    to_file: path to write Markdown; if None, prints to stdout
    """
    C = _coefs_orient_to_wvn_rows(coeffs, nu_cm1)
    nnu = len(nu_cm1)

    if sample is None:
        idxs = np.arange(nnu)
    elif isinstance(sample, int):
        idxs = np.linspace(0, nnu-1, sample, dtype=int)
    else:
        idxs = np.asarray(list(sample), dtype=int)

    lines = []
    lines.append(f"### Fitted log–polynomial for {species}\n")
    lines.append("Model:  \n"
                 r"`ln κ_m(ν;T,p) = a0 + a1 ln p + a2 ln T + a3 (ln p)^2 + a4 (ln T)^2 + a5 (ln p)(ln T)`  ")
    lines.append("_Units: κ_m in m²/kg, p in Pa, T in K._\n")
    for j in idxs:
        lines.append(f"- {format_logpoly6_equation_row(C[j], nu_cm1[j], species, cfmt=cfmt)}")

    md = "\n".join(lines)
    if to_file:
        with open(to_file, "w", encoding="utf-8") as f:
            f.write(md + "\n")
    else:
        print(md)


In [3]:
#load data

with np.load('kappas_pred_O3_PLS.npz') as data:
    kappas_pred_O3_PLS = data["data1"]

with np.load('kappas_pred_CO2_PLS.npz') as data:
    kappas_pred_CO2_PLS = data["data1"]

with np.load('kappas_pred_O3_temperature_predictor_only.npz') as data:
    kappas_pred_O3_temp_only = data["data1"]


with np.load('kappas_pred_CO2_temperature_predictor_only.npz') as data:
    kappas_pred_CO2_temp_only = data["data1"]

with np.load('kappas_pred_O3_pressure_predictor_only.npz') as data:
    kappas_pred_O3_pres_only = data["data1"]

with np.load('kappas_pred_CO2_pressure_predictor_only.npz') as data:
    kappas_pred_CO2_pres_only = data["data1"]

with np.load('O3_kappas_power1_pres.npz') as data:
    O3_kappas_power1_pres = data["data1"]

with np.load('O3_kappas_power1_temp.npz') as data:
    O3_kappas_power1_temp = data["data1"]
    
with np.load('CO2_kappas_power1_pres.npz') as data:
    CO2_kappas_power1_pres = data["data1"]

with np.load('CO2_kappas_power1_temp.npz') as data:
    CO2_kappas_power1_temp = data["data1"]

kappas_pred_PLS = {"CO2": kappas_pred_CO2_PLS[:,0:50,:], "O3" : kappas_pred_O3_PLS[:,0:50,:]}
kappas_pred_temp_only = {"CO2": kappas_pred_CO2_temp_only[:,0:50,:], "O3": kappas_pred_O3_temp_only[:,0:50,:]}
kappas_pred_pres_only = {"CO2": kappas_pred_CO2_pres_only[:,0:50,:], "O3": kappas_pred_O3_pres_only[:,0:50,:]}
kappas_power1_pres = {"CO2": CO2_kappas_power1_pres[:,0:50,:], "O3": O3_kappas_power1_pres[:,0:50,:]}
kappas_power1_temp = {"CO2": CO2_kappas_power1_temp[:,0:50,:], "O3": O3_kappas_power1_temp[:,0:50,:]}

In [4]:
print(f"kappas_pred_CO2_PLS shape is: {kappas_pred_CO2_PLS.shape}")
print(f"kappas_pred_O3_PLS shape is: {kappas_pred_CO2_PLS.shape}")
print(f"kappas_pred_CO2_temp_only shape is: {kappas_pred_CO2_temp_only.shape}")
print(f"kappas_pred_O3_temp_only shape is: {kappas_pred_CO2_temp_only.shape}")
print(f"kappas_pred_CO2_pres_only shape is: {kappas_pred_CO2_pres_only.shape}")
print(f"kappas_pred_O3_pres_only shape is: {kappas_pred_CO2_pres_only.shape}")
print(f"O3_kappas_power1_pres shape is: {O3_kappas_power1_pres.shape}")
print(f"O3_kappas_power1_temp shape is: {O3_kappas_power1_temp.shape}")
print(f"CO2_kappas_power1_pres shape is: {CO2_kappas_power1_pres.shape}")
print(f"CO2_kappas_power1_temp shape is: {CO2_kappas_power1_temp.shape}")

kappas_pred_CO2_PLS shape is: (64, 100, 55)
kappas_pred_O3_PLS shape is: (64, 100, 55)
kappas_pred_CO2_temp_only shape is: (64, 100, 55)
kappas_pred_O3_temp_only shape is: (64, 100, 55)
kappas_pred_CO2_pres_only shape is: (64, 100, 55)
kappas_pred_O3_pres_only shape is: (64, 100, 55)
O3_kappas_power1_pres shape is: (64, 100, 55)
O3_kappas_power1_temp shape is: (64, 100, 55)
CO2_kappas_power1_pres shape is: (64, 100, 55)
CO2_kappas_power1_temp shape is: (64, 100, 55)


## Global Cost

## CKDMIP Column Workflow

For each of the 50 CKDMIP columns:

* Build the full-mixture LBL reference on the DDQ grid (all gases).
* Build the surrogate mixture for that same column:
    * Swap CO₂ and/or O₃ to the $\kappa_m(T,p)$ surrogate.
    * Keep all other gases’ k from the LBL run.
* Extract:
    * `F_ref(column, level)` and `H_ref(column, level)` from DDQ LBL
    * `F_est(column, level)` and `H_est(column, level)` from surrogate

---

## Global Cost Calculation

Then combine everything and compute **one global cost**:

* Stack all flux errors across all levels and all columns into one big vector.
* Same for heating-rate errors (and later forcings).
* Apply the existing cost formula (with `F_ARRAY` weights) **once** to those stacked errors.
* **No** extra “normalize the 50 costs” step.

### The RMS itself is the normalization:

$$
\text{RMS} = \sqrt{\frac{1}{N}\sum_{i=1}^N e_i^2}
$$

---

### Conceptually:

* Right now I have run with `N_COLS = 1`.
* Next step: run with `N_COLS = 50` and build `y_ref`, `x_sup`, and `y_hat` that include all columns.

### Roughly:

1.  Loop `col = 0..49`:
    * `y_ref_col`, `x_sup_col`, (`gas_names`, `k_species_lbl`, `p_hl`, `T_hl`, `z`, `vmr`) from `build_reference_targets_from_lbl_fullmix` (which now uses the cache).
    * `y_hat_col` from `make_y_hat_with_species_swaps` for that column.
2.  Store those into the right slices of big arrays:
    * `reference_fluxes[level, col]`
    * `reference_heating[col, level]`
    * etc.
3.  ...and concatenate all `y_hat_col` flux pieces in the same flattening scheme that `cost_no_forcing` expects.
4.  Once I have:
    * `y_ref_all` (Dataset with all columns),
    * `x_sup_all` (pressures & heating for all columns),
    * `y_hat_all` (flattened over all levels $\times$ all columns),
5.  ...call:
    * `cost_global = cost_no_forcing(y_hat_all, y_ref_all, x_sup_all)`

Now `cost_global` is:

* one scalar,
* an RMS-style measure
* over all 50 profiles and all levels - the cost function includes all 50 columns

In [5]:
from tqdm.auto import tqdm


def _column_cache_path(cache_dir, col_idx):
    os.makedirs(cache_dir, exist_ok=True)
    return os.path.join(cache_dir, f"lbl_col_{col_idx:03d}.npz")


def load_column_lbl_cache(cache_dir, col_idx, nu_cm1):
    """
    Try to load cached full-mixture LBL data for one column.

    Returns:
        None if no valid cache.
        Otherwise:
          gas_names      : list[str]
          k_species_lbl  : (n_spec, n_wvn, n_level)
          p_hl           : (n_level,)
          T_hl           : (n_level,)
          z_levels       : (n_level,)
          vmr_dict       : {name: (n_level,)}
          F_ref_col      : (n_level,) net flux
          H_ref_col      : (n_level,) heating
          P_col          : (n_level,) pressures (same as p_hl)
    """
    path = _column_cache_path(cache_dir, col_idx)
    if not os.path.exists(path):
        return None

    data = np.load(path, allow_pickle=True)
    if "nu_cm1" not in data:
        return None

    nu_saved = np.array(data["nu_cm1"], dtype=float)
    nu_now = np.array(nu_cm1, dtype=float)
    if nu_saved.shape != nu_now.shape or np.max(np.abs(nu_saved - nu_now)) > 1e-9:
        # Different spectral grid -> ignore cache
        return None

    gas_names = [str(s) for s in data["gas_names"]]
    k_species_lbl = np.array(data["k_species_lbl"], dtype=float)
    p_hl = np.array(data["p_hl"], dtype=float)
    T_hl = np.array(data["T_hl"], dtype=float)
    z_levels = np.array(data["z_levels"], dtype=float)

    vmr_names = [str(s) for s in data["vmr_names"]]
    vmr_matrix = np.array(data["vmr_matrix"], dtype=float)
    vmr_dict = {name: vmr_matrix[i, :] for i, name in enumerate(vmr_names)}

    F_ref_col = np.array(data["F_ref_col"], dtype=float)
    H_ref_col = np.array(data["H_ref_col"], dtype=float)
    P_col = np.array(data["P_col"], dtype=float)

    return (
        gas_names,
        k_species_lbl,
        p_hl,
        T_hl,
        z_levels,
        vmr_dict,
        F_ref_col,
        H_ref_col,
        P_col,
    )


def save_column_lbl_cache(
    cache_dir,
    col_idx,
    nu_cm1,
    gas_names,
    k_species_lbl,
    p_hl,
    T_hl,
    z_levels,
    vmr_dict,
    F_ref_col,
    H_ref_col,
    P_col,
):
    """
    Save everything needed to reconstruct both:
      - the full-mixture LBL reference for this column, and
      - the per-species info needed to build surrogate mixtures later.
    """
    path = _column_cache_path(cache_dir, col_idx)
    os.makedirs(cache_dir, exist_ok=True)

    vmr_names = np.array(list(vmr_dict.keys()))
    vmr_matrix = np.vstack([vmr_dict[name] for name in vmr_names])

    np.savez_compressed(
        path,
        nu_cm1=np.array(nu_cm1, dtype=float),
        gas_names=np.array(gas_names),
        k_species_lbl=np.array(k_species_lbl, dtype=float),
        p_hl=np.array(p_hl, dtype=float),
        T_hl=np.array(T_hl, dtype=float),
        z_levels=np.array(z_levels, dtype=float),
        vmr_names=vmr_names,
        vmr_matrix=vmr_matrix,
        F_ref_col=np.array(F_ref_col, dtype=float),
        H_ref_col=np.array(H_ref_col, dtype=float),
        P_col=np.array(P_col, dtype=float),
    )


def compute_column_reference_and_cache(
    col_idx,
    profile_col,
    nu_cm1,
    weights,
    N_LEVELS,
    mat,
    cache_dir,
):
    """
    Run full-mixture LBL + two-stream for one column, then cache results.

    Returns same tuple as load_column_lbl_cache.
    """
    (
        y_ref_col,
        x_sup_col,
        (gas_names,
         k_species_lbl,
         p_hl,
         T_hl,
         z_levels,
         vmr_dict),
    ) = build_reference_targets_from_lbl_fullmix(
        profile_col,
        nu_cm1=nu_cm1,
        weights=weights,
        N_LEVELS=N_LEVELS,
        mat=mat,
        N_COLS=1,
        N_SCENARIOS=1,
    )

    F_ref_col = np.asarray(y_ref_col["reference_fluxes"].values[:, 0], float)
    H_ref_col = np.asarray(x_sup_col["reference_heating"].values[0, :], float)
    P_col     = np.asarray(x_sup_col["pressures"].values[0, :], float)

    save_column_lbl_cache(
        cache_dir,
        col_idx,
        nu_cm1,
        gas_names,
        k_species_lbl,
        p_hl,
        T_hl,
        z_levels,
        vmr_dict,
        F_ref_col,
        H_ref_col,
        P_col,
    )

    return (
        gas_names,
        k_species_lbl,
        p_hl,
        T_hl,
        z_levels,
        vmr_dict,
        F_ref_col,
        H_ref_col,
        P_col,
    )



def compute_global_cost_over_all_columns(
    profiles,
    DDQ_lw,
    kappa_coeffs_dict,
    molar_mass_dict,
    target_species_list=("CO2", "O3"),
    cache_dir="lbl_cache",
):
    """
    Compute ONE global cost over all CKDMIP columns.

    For each column:
      - Try to load cached full-mixture LBL reference and species info.
      - If missing, run full LBL + two-stream once, then cache it.
      - Build surrogate mixture (CO2/O3 swapped to κ_m fits) for that column.
    Then:
      - Stack net fluxes across all columns into y_hat_all.
      - Stack reference flux+heating into y_ref_all/x_sup_all.
      - Call cost_no_forcing once.

    Returns:
      cost_global : float
    """
    nu_cm1  = np.asarray(DDQ_lw.S.values, float)
    weights = np.asarray(DDQ_lw.W.values, float)

    n_cols = int(profiles.dims["column"])
    n_lev  = int(profiles.dims["half_level"])

    # These must match cost_no_forcing
    global N_LEVELS, N_COLS, N_SCENARIOS, mat
    N_LEVELS    = n_lev
    N_COLS      = n_cols
    N_SCENARIOS = 1
    mat         = derivative_matrix(N_LEVELS)

    # Allocate global arrays
    F_ref_all = np.zeros((N_LEVELS, N_COLS))
    H_ref_all = np.zeros((N_COLS, N_LEVELS))
    P_all     = np.zeros((N_COLS, N_LEVELS))
    F_est_all = np.zeros((N_LEVELS, N_COLS))

    iterator = range(N_COLS)
    if tqdm is not None:
        iterator = tqdm(iterator, desc="Processing CKDMIP columns", ncols=80)

    for col in iterator:
        profile_col = profiles.isel(column=col)

        # 1. Try disk cache
        cached = load_column_lbl_cache(cache_dir, col, nu_cm1)

        if cached is None:
            # No valid cache; compute and save
            print(f"[col {col}] cache miss -> running full LBL", flush=True)
            (
                gas_names,
                k_species_lbl,
                p_hl,
                T_hl,
                z_levels,
                vmr_dict,
                F_ref_col,
                H_ref_col,
                P_col,
            ) = compute_column_reference_and_cache(
                col,
                profile_col,
                nu_cm1,
                weights,
                N_LEVELS,
                mat,
                cache_dir,
            )
        else:
            # Use cached LBL + reference
            (
                gas_names,
                k_species_lbl,
                p_hl,
                T_hl,
                z_levels,
                vmr_dict,
                F_ref_col,
                H_ref_col,
                P_col,
            ) = cached

        # Store reference into global arrays
        F_ref_all[:, col] = F_ref_col
        H_ref_all[col, :] = H_ref_col
        P_all[col, :]     = P_col

        # 2. Build surrogate mixture for this column
        y_hat_col = make_y_hat_with_species_swaps(
            target_species_list=target_species_list,
            gas_names=gas_names,
            k_species_lbl=k_species_lbl,
            p_hl=p_hl,
            T_hl=T_hl,
            z_levels=z_levels,
            nu_cm1=nu_cm1,
            weights=weights,
            kappa_coeffs_dict=kappa_coeffs_dict,
            molar_mass_dict=molar_mass_dict,
            vmr_dict=vmr_dict,
            col=col
        )

        # First N_LEVELS entries correspond to net flux for this column
        F_est_all[:, col] = np.asarray(y_hat_col[:N_LEVELS], float)

    # 3. Build global reference / supplementary datasets in cost_no_forcing format

    level_coord  = np.arange(N_LEVELS)
    column_coord = np.arange(N_COLS)

    y_ref_all = xr.Dataset(
        data_vars=dict(
            reference_fluxes       =(("level", "column"), F_ref_all),
            reference_forcing_co2  =(("column",), np.zeros(N_COLS)),
            reference_forcing_ch4  =(("column",), np.zeros(N_COLS)),
            reference_forcing_n2o  =(("column",), np.zeros(N_COLS)),
            reference_forcing_o3   =(("column",), np.zeros(N_COLS)),
            reference_forcing_cfc11=(("column",), np.zeros(N_COLS)),
            reference_forcing_cfc12=(("column",), np.zeros(N_COLS)),
        ),
        coords=dict(level=level_coord, column=column_coord),
    )

    x_sup_all = xr.Dataset(
        data_vars=dict(
            pressures        =(("column", "level"), P_all),
            reference_heating=(("column", "level"), H_ref_all),
        ),
        coords=dict(level=level_coord, column=column_coord),
    )

    # 4. Assemble y_hat over ALL columns:
    # fluxes flattened in Fortran order (levels fast, columns slow) as expected by cost_no_forcing
    F_est_flat = F_est_all.T.flatten(order="F")
    zeros_forc = np.zeros(6 * N_COLS * N_SCENARIOS)
    y_hat_all  = np.concatenate([F_est_flat, zeros_forc])

    # 5. One global cost
    cost_expr = cost_no_forcing(y_hat_all, y_ref_all, x_sup_all)
    try:
        cost_val = float(cost_expr.value)
    except AttributeError:
        cost_val = float(cost_expr)

    print(f"Global cost over all {N_COLS} columns (targets: {target_species_list}): {cost_val}")
    return cost_val

/srv/conda/envs/ddq_fluxsim_tutorial/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
profiles_full = xr.open_dataset("ckdmip_evaluation1_concentrations_present.nc")
N_LEVELS = profiles_full.pressure_hl.size
# Load sparse wavenumbers
DDQ_lw = xr.open_dataset("DDQ_LW_present.h5", engine="netcdf4")
DDQ_lw.compute()
wvn_sparse = DDQ_lw.S.values.astype(float)

N_SCENARIOS = 1
N_COLS = 1
F_ARRAY = np.array([0.15, 1, 1, 1, 1, 1, 1, 1])

mat = derivative_matrix(N_LEVELS)



## CKMDIP Evaluation Dataset 1 as the predictions

In [7]:
# PLS Cost Eval:

print(f"PLS cost evaluation")

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_PLS,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),   # or ("CO2",) or ("O3",)
)

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_PLS,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2",),   # or ("CO2",) or ("O3",)
)
global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_PLS,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("O3",),   # or ("CO2",) or ("O3",)
)


/tmp/ipykernel_5620/3824427884.py:200: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_cols = int(profiles.dims["column"])
/tmp/ipykernel_5620/3824427884.py:201: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lev  = int(profiles.dims["half_level"])


PLS cost evaluation


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 150.11it/s]


Global cost over all 50 columns (targets: ('CO2', 'O3')): 208.7358331759629


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 158.12it/s]


Global cost over all 50 columns (targets: ('CO2',)): 250.20313057251428


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 164.49it/s]

Global cost over all 50 columns (targets: ('O3',)): 89.34773393156202


In [8]:
# Temp Only Cost Eval:

print(f"Temp only cost evaluation")

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_temp_only,   # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),   # or ("CO2",) or ("O3",)
)

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_temp_only,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2",),   # or ("CO2",) or ("O3",)
)
global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_temp_only,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("O3",),   # or ("CO2",) or ("O3",)
)


/tmp/ipykernel_5620/3824427884.py:200: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_cols = int(profiles.dims["column"])
/tmp/ipykernel_5620/3824427884.py:201: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lev  = int(profiles.dims["half_level"])


Temp only cost evaluation


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 142.01it/s]


Global cost over all 50 columns (targets: ('CO2', 'O3')): 226.1092846297557


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 159.09it/s]


Global cost over all 50 columns (targets: ('CO2',)): 253.36653432475939


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 159.34it/s]

Global cost over all 50 columns (targets: ('O3',)): 89.29243108376632


In [9]:
# Pressure Only Cost Eval:

print(f"Pres only cost evaluation")

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_pres_only,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),   # or ("CO2",) or ("O3",)
)

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_pres_only,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2",),   # or ("CO2",) or ("O3",)
)
global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_pres_only,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("O3",),   # or ("CO2",) or ("O3",)
)


/tmp/ipykernel_5620/3824427884.py:200: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_cols = int(profiles.dims["column"])
/tmp/ipykernel_5620/3824427884.py:201: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lev  = int(profiles.dims["half_level"])


Pres only cost evaluation


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 158.13it/s]


Global cost over all 50 columns (targets: ('CO2', 'O3')): 208.4246975104913


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 161.91it/s]


Global cost over all 50 columns (targets: ('CO2',)): 249.84152949476334


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 157.98it/s]

Global cost over all 50 columns (targets: ('O3',)): 89.34249240763444


In [13]:
# power1_pres Linear eval:

print(f"Linear power 1 pressure only eval")

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_pres,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),   # or ("CO2",) or ("O3",)
)

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_pres,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2",),   # or ("CO2",) or ("O3",)
)
global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_pres,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("O3",),   # or ("CO2",) or ("O3",)
)

/tmp/ipykernel_5620/3824427884.py:200: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_cols = int(profiles.dims["column"])
/tmp/ipykernel_5620/3824427884.py:201: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lev  = int(profiles.dims["half_level"])


Linear power 1 pressure only eval


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 157.71it/s]


Global cost over all 50 columns (targets: ('CO2', 'O3')): 454133.79731106677


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 113.51it/s]


Global cost over all 50 columns (targets: ('CO2',)): 414180.0776170118


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 148.01it/s]

Global cost over all 50 columns (targets: ('O3',)): 104.36481912767725


In [14]:
# power1_temp linear eval

print(f"Linear power 1 temperature only eval")

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_temp,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),   # or ("CO2",) or ("O3",)
)

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_temp,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2",),   # or ("CO2",) or ("O3",)
)
global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_temp,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("O3",),   # or ("CO2",) or ("O3",)
)



/tmp/ipykernel_5620/3824427884.py:200: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_cols = int(profiles.dims["column"])
/tmp/ipykernel_5620/3824427884.py:201: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lev  = int(profiles.dims["half_level"])


Linear power 1 temperature only eval


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 161.47it/s]


Global cost over all 50 columns (targets: ('CO2', 'O3')): 141953.24480439402


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 161.46it/s]


Global cost over all 50 columns (targets: ('CO2',)): 130530.62546590829


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 158.94it/s]

Global cost over all 50 columns (targets: ('O3',)): 104.1662046530072


## CKDMIP Evaluation 2 used for predictions:

In [15]:
# select the predicitons for evaluation 2 dataset
kappas_pred_PLS = {"CO2": kappas_pred_CO2_PLS[:,50:,:], "O3" : kappas_pred_O3_PLS[:,50:,:]}
kappas_pred_temp_only = {"CO2": kappas_pred_CO2_temp_only[:,50:,:], "O3": kappas_pred_O3_temp_only[:,50:,:]}
kappas_pred_pres_only = {"CO2": kappas_pred_CO2_pres_only[:,50:,:], "O3": kappas_pred_O3_pres_only[:,50:,:]}
kappas_power1_pres = {"CO2": CO2_kappas_power1_pres[:,50:,:], "O3": O3_kappas_power1_pres[:,50:,:]}
kappas_power1_temp = {"CO2": CO2_kappas_power1_temp[:,50:,:], "O3": O3_kappas_power1_temp[:,50:,:]}

In [16]:
profiles_full = xr.open_dataset("ckdmip_evaluation2_concentrations_present.nc")
N_LEVELS = profiles_full.pressure_hl.size
# Load sparse wavenumbers
DDQ_lw = xr.open_dataset("DDQ_LW_present.h5", engine="netcdf4")
DDQ_lw.compute()
wvn_sparse = DDQ_lw.S.values.astype(float)

N_SCENARIOS = 1
N_COLS = 1
F_ARRAY = np.array([0.15, 1, 1, 1, 1, 1, 1, 1])

mat = derivative_matrix(N_LEVELS)


In [17]:
# PLS Cost Eval:

print(f"PLS cost evaluation")

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_PLS,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),   # or ("CO2",) or ("O3",)
)

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_PLS,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2",),   # or ("CO2",) or ("O3",)
)
global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_PLS,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("O3",),   # or ("CO2",) or ("O3",)
)


/tmp/ipykernel_5620/3824427884.py:200: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_cols = int(profiles.dims["column"])
/tmp/ipykernel_5620/3824427884.py:201: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lev  = int(profiles.dims["half_level"])


PLS cost evaluation


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 157.92it/s]


Global cost over all 50 columns (targets: ('CO2', 'O3')): 209.3330882181698


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 155.98it/s]


Global cost over all 50 columns (targets: ('CO2',)): 251.01972467481747


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 159.18it/s]

Global cost over all 50 columns (targets: ('O3',)): 89.34863610553998


In [18]:
# Temp Only Cost Eval:

print(f"Temp only cost evaluation")

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_temp_only,   # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),   # or ("CO2",) or ("O3",)
)

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_temp_only,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2",),   # or ("CO2",) or ("O3",)
)
global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_temp_only,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("O3",),   # or ("CO2",) or ("O3",)
)


/tmp/ipykernel_5620/3824427884.py:200: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_cols = int(profiles.dims["column"])
/tmp/ipykernel_5620/3824427884.py:201: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lev  = int(profiles.dims["half_level"])


Temp only cost evaluation


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 156.04it/s]


Global cost over all 50 columns (targets: ('CO2', 'O3')): 226.68410534694337


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 132.79it/s]


Global cost over all 50 columns (targets: ('CO2',)): 254.1861315097508


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 155.56it/s]

Global cost over all 50 columns (targets: ('O3',)): 89.2920060287143


In [19]:
# Pressure Only Cost Eval:

print(f"Pres only cost evaluation")

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_pres_only,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),   # or ("CO2",) or ("O3",)
)

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_pres_only,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2",),   # or ("CO2",) or ("O3",)
)
global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_pred_pres_only,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("O3",),   # or ("CO2",) or ("O3",)
)


/tmp/ipykernel_5620/3824427884.py:200: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_cols = int(profiles.dims["column"])
/tmp/ipykernel_5620/3824427884.py:201: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lev  = int(profiles.dims["half_level"])


Pres only cost evaluation


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 149.86it/s]


Global cost over all 50 columns (targets: ('CO2', 'O3')): 208.31394842601665


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 153.31it/s]


Global cost over all 50 columns (targets: ('CO2',)): 249.8149971086201


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 155.75it/s]

Global cost over all 50 columns (targets: ('O3',)): 89.34373626003722


In [20]:
# power1_pres Linear eval:

print(f"Linear power 1 pressure only eval")

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_pres,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),   # or ("CO2",) or ("O3",)
)

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_pres,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2",),   # or ("CO2",) or ("O3",)
)
global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_pres,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("O3",),   # or ("CO2",) or ("O3",)
)


/tmp/ipykernel_5620/3824427884.py:200: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_cols = int(profiles.dims["column"])
/tmp/ipykernel_5620/3824427884.py:201: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lev  = int(profiles.dims["half_level"])


Linear power 1 pressure only eval


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 153.01it/s]


Global cost over all 50 columns (targets: ('CO2', 'O3')): 454133.79731106677


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 151.61it/s]


Global cost over all 50 columns (targets: ('CO2',)): 414180.0776170118


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 153.42it/s]

Global cost over all 50 columns (targets: ('O3',)): 104.36481912767725


In [21]:
# power1_temp linear eval

print(f"Linear power 1 temperature only eval")

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_temp,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2", "O3"),   # or ("CO2",) or ("O3",)
)

global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_temp,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("CO2",),   # or ("CO2",) or ("O3",)
)
global_cost = compute_global_cost_over_all_columns(
    profiles=profiles_full,
    DDQ_lw=DDQ_lw,
    kappa_coeffs_dict=kappas_power1_temp,      # fitted CO2/O3 κ_m coefficients
    molar_mass_dict=MOLAR_MASS,
    target_species_list=("O3",),   # or ("CO2",) or ("O3",)
)



/tmp/ipykernel_5620/3824427884.py:200: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_cols = int(profiles.dims["column"])
/tmp/ipykernel_5620/3824427884.py:201: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_lev  = int(profiles.dims["half_level"])


Linear power 1 temperature only eval


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 152.60it/s]


Global cost over all 50 columns (targets: ('CO2', 'O3')): 141953.24480439402


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 155.60it/s]


Global cost over all 50 columns (targets: ('CO2',)): 130530.62546590829


Processing CKDMIP columns: 100%|███████████████| 50/50 [00:00<00:00, 152.18it/s]

Global cost over all 50 columns (targets: ('O3',)): 104.1662046530072
